# Reproducing the dissertation's tables and stats

This notebook rebuilds every table in the dissertation from the CSVs
already in this repo. No retraining, no raw data needed, runs in under a
minute.

It's not the full pipeline -- see `RUNBOOK.md` for that (~33h from raw
data). This just checks the dissertation's numbers against the results
files.

Needs `pandas`, `numpy`, `scipy` (already in `requirements.txt`). Run from
the repo root.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

ROOT = Path(".")
assert (ROOT / "RUNBOOK.md").exists(), (
    "Run this notebook from the repository root (where RUNBOOK.md lives)."
)

CHECKS = []  # every check we run, for the summary table at the end

def record(label, diss_value, computed_value, note=""):
    CHECKS.append({"check": label, "dissertation": diss_value,
                    "computed": computed_value, "note": note})

def record_numeric(label, diss_value, computed_value):
    # same as record(), but flags a mismatch automatically
    note = "" if computed_value == diss_value else "MISMATCH"
    record(label, diss_value, computed_value, note)


## Where each file comes from

How the raw data turns into `stops_features_osm.csv`, step by step (full
detail in `RUNBOOK.md`).

In [ ]:
pipeline = pd.DataFrame([
    ("step1_aggregate_busto.py",        "data/*Weekday*QUARTER HOUR*.csv (raw BUSTO)", "busto_stop_level_boardings.csv",
     "sums boardings per stop; also counts service coverage"),
    ("step2_join_coordinates.py",       "+ data/Bus_Stops.csv",                        "stops_with_coords.csv",
     "joins stop coordinates on STOPCODE"),
    ("step3_lsoa_features.py",          "+ data/access_*.csv (AI23)",                  "stops_features.csv",
     "postcode -> LSOA -> joins the 8 AI23 columns, filters to London -> 17,943 stops"),
    ("step3b_osm_features.py",          "+ live Overpass API",                         "stops_features_osm.csv",
     "adds 6 OSM point-of-interest columns (500m buffer)"),
    ("step3c_add_scenic.py",            "(same file)",                                 "stops_features_osm.csv",
     "adds the 7th POI category, poi_scenic"),
    ("step3d_add_service_coverage.py",  "+ busto_stop_level_boardings.csv",            "stops_features_osm.csv",
     "merges in service_coverage -- this is the file every step4* script reads"),
], columns=["script", "reads", "writes", "what it does"])
pipeline


## Which command makes which result

Copy-paste any row to reproduce it. One thing to know first:
`step4_model.py` always trains all 7 models together, so a single run
takes the full ~1.5-2h even if you only care about Random Forest. **For a
fast, tabular-only check, use `step4c_fast_baselines.py` or
`step4j_tuned_baselines.py` instead** -- both skip GATv2/MLP.

In [ ]:
commands = pd.DataFrame([
    ("AI23 (table column)",           "python step4c_fast_baselines.py",              "results_cv_ai23_only_fastbaselines.csv",  "tabular only, fast"),
    ("OSM (table column)",            "python step4c_fast_baselines.py",              "results_cv_osm_only_fastbaselines.csv",   "same script writes all 3"),
    ("AI23+OSM (table column)",       "python step4c_fast_baselines.py",              "results_cv_ai23_osm_fastbaselines.csv",   "same script writes all 3"),
    ("AI23+SC",                       "python -u step4_model.py --ai23-only --with-sc","results_cv_ai23_sc.csv",                  "full suite, ~1.5-2h"),
    ("AI23+OSM+SC (HEADLINE)",        "python -u step4_model.py --with-sc",           "results_cv_ai23_osm_sc.csv",              "full suite, ~1.5-2h"),
    ("Ridge/RF/XGBoost tuned",        "python step4j_tuned_baselines.py",             "results_cv_tuned.csv",                    "tabular only, ~39s/fold"),
    ("IDW baseline",                  "python step4d_idw_baseline.py",                "results_cv_idw.csv",                      "~1 sec total"),
    ("GATv2, K=10",                   "python -u step4_model.py --with-sc --k10",     "results_cv_gnn_fairness.csv",             "full suite, ~1.5-2h"),
    ("GATv2, func-sim edges",         "python -u step4_model.py --with-sc --func-sim","results_cv_func_sim.csv",                 "full suite, ~1.5-2h"),
    ("PCA-AI23 robustness check",     "python -u step4_model.py --with-sc --pca-ai23","results_cv_pca_ai23_osm_sc.csv",          "full suite, ~1.5-2h"),
    ("GATv2-Fusion",                  "python step4e_zheng_fusion.py",                "results_cv_zheng_fusion.csv",             "imports graph builders from step4_model.py"),
    ("GCN",                           "python step4i_gcn_baseline.py",                "results_cv_gcn.csv",                      "GATv2Conv swapped for GCNConv"),
    ("5-seed MLP noise floor",        "python step4l_multiseed_mlp.py",               "results_summary_multiseed_mlp.csv",       "standalone MLP, 5 seeds"),
], columns=["dissertation column/row", "command", "output file", "notes"])
pd.set_option("display.max_colwidth", 80)
commands


## 1. Main results table

The headline table: WMAPE for every model, across every feature set.

One gap to work around: `results_summary_ai23_sc.csv` isn't in the merged
file yet (`merge_results.py` didn't know about that config last time it
ran), so this notebook reads it separately.

In [ ]:
all_summary = pd.read_csv("all_results_summary.csv")
ai23_sc_summary = pd.read_csv("results_summary_ai23_sc.csv")
ai23_sc_summary.insert(0, "config", "ai23_sc")

def wmape(config, model, df=all_summary):
    row = df[(df["config"] == config) & (df["model"] == model)]
    if row.empty:
        return None
    return round(float(row["WMAPE_mean"].iloc[0]), 4)

COLUMNS = {
    "AI23":         "ai23_only_fastbaselines",
    "OSM":          "osm_only_fastbaselines",
    "AI23+OSM":     "ai23_osm_fastbaselines",
    "AI23+SC":      None,   # handled specially, standalone file
    "AI23+OSM+SC":  "ai23_osm_sc",
}

# row label, model name in the CSV, which columns it has a value for
ROWS = [
    ("Historical average", "HistAvg",  ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("IDW (no features)",  "IDW",      ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("Ridge (a=1.0)",      "MLR",      ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("Random forest",      "RF",       ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("XGBoost",            "XGBoost",  ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]),
    ("MLP",                "MLP",      ["AI23+SC", "AI23+OSM+SC"]),
    ("GATv2 (K=5)",        "GATv2",    ["AI23+SC", "AI23+OSM+SC"]),
]

table = {}
for label, model, present_cols in ROWS:
    row = {}
    for col in ["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]:
        if col not in present_cols:
            row[col] = "—"
            continue
        if model == "IDW":
            # IDW doesn't use features, so just reuse one value everywhere
            row[col] = wmape("idw", "IDW")
        elif col == "AI23+SC":
            row[col] = wmape("ai23_sc", model, df=ai23_sc_summary)
        else:
            row[col] = wmape(COLUMNS[col], model)
    table[label] = row

main_results = pd.DataFrame(table).T[["AI23", "OSM", "AI23+OSM", "AI23+SC", "AI23+OSM+SC"]]
main_results


In [ ]:
# tuned baselines and graph-model variants (AI23+OSM+SC column only)
extra_rows = {
    "Ridge tuned":          wmape("tuned", "MLR-tuned"),
    "Random forest tuned":  wmape("tuned", "RF-tuned"),
    "XGBoost tuned":        wmape("tuned", "XGBoost-tuned"),
    "GATv2, K=10":          wmape("gnn_fairness", "GATv2"),
    "GATv2-Fusion":         wmape("zheng_fusion", "GATv2-Fusion"),
    "GATv2, func-sim edges":wmape("func_sim", "GATv2"),
    "GCN":                  wmape("gcn", "GCN"),
}
extra = pd.Series(extra_rows, name="AI23+OSM+SC")
extra_df = extra.to_frame()
extra_df


In [ ]:
# check every cell above against the dissertation's numbers
DISSERTATION_MAIN = {
    ("Historical average", "AI23"): 1.0822, ("Historical average", "OSM"): 1.0822,
    ("Historical average", "AI23+OSM"): 1.0822, ("Historical average", "AI23+SC"): 1.0822,
    ("Historical average", "AI23+OSM+SC"): 1.0822,
    ("IDW (no features)", "AI23"): 0.8487, ("IDW (no features)", "OSM"): 0.8487,
    ("IDW (no features)", "AI23+OSM"): 0.8487, ("IDW (no features)", "AI23+SC"): 0.8487,
    ("IDW (no features)", "AI23+OSM+SC"): 0.8487,
    ("Ridge (a=1.0)", "AI23"): 0.8120, ("Ridge (a=1.0)", "OSM"): 0.8078,
    ("Ridge (a=1.0)", "AI23+OSM"): 0.7982, ("Ridge (a=1.0)", "AI23+SC"): 0.6420,
    ("Ridge (a=1.0)", "AI23+OSM+SC"): 0.6404,
    ("Random forest", "AI23"): 0.8128, ("Random forest", "OSM"): 0.8064,
    ("Random forest", "AI23+OSM"): 0.7970, ("Random forest", "AI23+SC"): 0.6452,
    ("Random forest", "AI23+OSM+SC"): 0.6428,
    ("XGBoost", "AI23"): 0.8207, ("XGBoost", "OSM"): 0.8084,
    ("XGBoost", "AI23+OSM"): 0.8075, ("XGBoost", "AI23+SC"): 0.6584,
    ("XGBoost", "AI23+OSM+SC"): 0.6437,
    ("MLP", "AI23+SC"): 0.6365, ("MLP", "AI23+OSM+SC"): 0.6311,
    ("GATv2 (K=5)", "AI23+SC"): 0.7262, ("GATv2 (K=5)", "AI23+OSM+SC"): 0.7187,
}
DISSERTATION_EXTRA = {
    "Ridge tuned": 0.6401, "Random forest tuned": 0.6339, "XGBoost tuned": 0.6497,
    "GATv2, K=10": 0.7372, "GATv2-Fusion": 0.7070, "GATv2, func-sim edges": 0.7352,
    "GCN": 0.7006,
}

n_mismatch = 0
for (row, col), diss_val in DISSERTATION_MAIN.items():
    got = table[row][col]
    match = (got == diss_val)
    n_mismatch += not match
    record(f"Main results: {row} / {col}", diss_val, got, "" if match else "MISMATCH")
for row, diss_val in DISSERTATION_EXTRA.items():
    got = extra_rows[row]
    match = (got == diss_val)
    n_mismatch += not match
    record(f"Main results: {row}", diss_val, got, "" if match else "MISMATCH")

print(f"Main results table: {len(DISSERTATION_MAIN) + len(DISSERTATION_EXTRA)} cells checked, "
      f"{n_mismatch} mismatch(es).")


## 2. Five-seed MLP check

The dissertation reports MLP = 0.6311 from one run, and 0.6301 +/- 0.0004
across 5 seeds, to show the headline result is stable.

In [ ]:
seeds = pd.read_csv("results_summary_multiseed_mlp.csv")
mean_wmape = seeds["WMAPE_mean"].mean()
std_wmape = seeds["WMAPE_mean"].std()   # sample std, same convention as the dissertation
print(seeds)
print(f"\nmean = {mean_wmape:.4f}, std = {std_wmape:.4f}")

record_numeric("5-seed MLP envelope (mean)", 0.6301, round(mean_wmape, 4))
record_numeric("5-seed MLP envelope (std)", 0.0004, round(std_wmape, 4))


## 3. Significance tests

These p-values used to only exist as output pasted into old log files --
no script actually computed them. This cell does.

In [ ]:
def paired_wilcoxon(csv_path, model_a, model_b):
    df = pd.read_csv(csv_path)
    a = df[df["model"] == model_a].set_index("borough")["WMAPE"].sort_index()
    b = df[df["model"] == model_b].set_index("borough")["WMAPE"].sort_index()
    assert list(a.index) == list(b.index), "borough sets don't align"
    diff = a - b   # positive means A worse (higher WMAPE) than B
    stat, p = stats.wilcoxon(a, b)
    return float(diff.mean()), float(p), len(a)

SIG_PAIRS = [
    ("GCN vs GATv2 (K=5)",        "results_cv_gcn.csv",          "GCN", "GATv2",       -0.0181, 0.010),
    ("GCN vs MLP",                "results_cv_gcn.csv",          "GCN", "MLP",          0.0695, 0.00001),
    ("MLP vs RF (tuned)",         "results_cv_tuned.csv",        "MLP", "RF-tuned",    -0.0028, 0.292),
    ("GATv2-Fusion vs GATv2(K=5)","results_cv_zheng_fusion.csv", "GATv2-Fusion", "GATv2", -0.0117, 0.126),
    ("Ridge vs RF (150)",         "results_cv_ai23_osm_sc.csv",  "MLR", "RF",          -0.0024, 0.357),
    # not in the significance table itself, but it's the number behind
    # fig3's p-value annotation -- worth checking too
    ("MLP vs GATv2 (K=5)",        "results_cv_ai23_osm_sc.csv",  "MLP", "GATv2",       -0.0876, 2.33e-10),
]

sig_rows = []
for label, path, a, b, diss_effect, diss_p in SIG_PAIRS:
    effect, p, n = paired_wilcoxon(path, a, b)
    sig_rows.append({"comparison": label, "effect_mean": round(effect, 4), "p_value": round(p, 5),
                      "n_folds": n, "dissertation_effect": diss_effect, "dissertation_p": diss_p})
    record_numeric(f"Significance: {label} (effect)", diss_effect, round(effect, 4))
    # tiny p-values are reported as thresholds (p<0.00001), so check against
    # that instead of an exact match
    p_ok = (p < 0.0001) if diss_p <= 0.00001 else abs(p - diss_p) < 0.01
    record(f"Significance: {label} (p-value, vs threshold)", diss_p, round(p, 5), "" if p_ok else "CHECK")

pd.DataFrame(sig_rows)


## 4. Service coverage vs. boardings

Pearson r, Spearman rho and R^2 between `service_coverage` and
`total_boardings`, both log1p-transformed, n=17,943.

In [ ]:
feat = pd.read_csv("stops_features_osm.csv")
assert len(feat) == 17943, f"expected 17,943 stops, got {len(feat)}"

x = np.log1p(feat["service_coverage"])
y = np.log1p(feat["total_boardings"])

pearson_r, _ = stats.pearsonr(x, y)
spearman_rho, _ = stats.spearmanr(x, y)
r2 = pearson_r ** 2

print(f"Pearson r  = {pearson_r:.4f}")
print(f"Spearman rho = {spearman_rho:.4f}")
print(f"R^2        = {r2:.4f}")

record_numeric("sc-univariate: Pearson r", 0.6882, round(pearson_r, 4))
record_numeric("sc-univariate: Spearman rho", 0.72, round(spearman_rho, 2))
record_numeric("sc-univariate: R^2", 0.47, round(r2, 2))


## 5. Feature ranges

Nothing in the repo computed this before -- the min/max table in the
dissertation was typed by hand. This is the first time it exists as code.

In [ ]:
FEATURE_COLS = [
    "employment_all_30min", "hospitals_30min", "gp_30min", "supermarkets_30min",
    "pharmacies_30min", "primary_schools_30min", "secondary_schools_30min", "main_bua_30min",
    "poi_residential", "poi_shopping", "poi_company", "poi_education", "poi_entertainment",
    "poi_scenic", "service_coverage", "lat", "lon",
]
ranges = feat[FEATURE_COLS].agg(["min", "max"]).T
ranges.columns = ["min", "max"]
ranges


In [ ]:
DISSERTATION_RANGES = {
    "employment_all_30min": (875, 2907645), "hospitals_30min": (0, 43), "gp_30min": (0, 218),
    "supermarkets_30min": (0, 91), "pharmacies_30min": (0, 411), "primary_schools_30min": (2, 289),
    "secondary_schools_30min": (0, 69), "main_bua_30min": (0, 1), "poi_residential": (0, 235),
    "poi_shopping": (0, 992), "poi_company": (0, 249), "poi_education": (0, 116),
    "poi_entertainment": (0, 958), "poi_scenic": (0, 187), "service_coverage": (2, 1504),
}
for col, (lo, hi) in DISSERTATION_RANGES.items():
    got_lo, got_hi = ranges.loc[col, "min"], ranges.loc[col, "max"]
    match = (got_lo == lo) and (got_hi == hi)
    record(f"Feature range: {col}", f"{lo}-{hi}", f"{got_lo:g}-{got_hi:g}", "" if match else "MISMATCH")
print("lat range:", ranges.loc["lat", "min"], "-", ranges.loc["lat", "max"], "  (dissertation: 51.2929-51.6846)")
print("lon range:", ranges.loc["lon", "min"], "-", ranges.loc["lon", "max"], "  (dissertation: -0.4995-0.2978)")


## 6. Target stats

Min/median/mean/max/skew of `total_boardings`, before and after log1p,
plus the zero-boarding count.

One thing this can't check: how many of the 287 zero-boarding stops also
have positive alightings. That needs the raw BUSTO data (not in this
repo), so it's not reproducible from what's committed here.

In [ ]:
y = feat["total_boardings"]
y_log = np.log1p(y)

target_stats = {
    "min": y.min(), "median": y.median(), "mean": y.mean(), "max": y.max(),
    "skew": stats.skew(y), "skew (log1p)": stats.skew(y_log),
}
for k, v in target_stats.items():
    print(f"{k:>14}: {v:.4f}" if isinstance(v, float) else f"{k:>14}: {v}")

n_zero = int((y == 0).sum())
pct_zero = 100 * n_zero / len(y)
print(f"\nzero-boarding stops: {n_zero} ({pct_zero:.1f}%)")

record_numeric("Target median", 112.6, round(float(target_stats["median"]), 1))
record_numeric("Target mean", 293.8, round(float(target_stats["mean"]), 1))
record_numeric("Target max", 14137, int(target_stats["max"]))
record_numeric("Target skewness", 5.64, round(float(target_stats["skew"]), 2))
record_numeric("Target skewness (log1p)", -0.55, round(float(target_stats["skew (log1p)"]), 2))
record_numeric("Zero-boarding stops (n)", 287, n_zero)
record_numeric("Zero-boarding stops (%)", 1.6, round(pct_zero, 1))


## 7. VIF and correlation check

Two numbers came out different when recomputed fresh:

- VIF: dissertation says 37.98 / 18.24, this gives 37.68 / 17.70
- Strongest AI23 pair: dissertation says 0.98, this gives 0.97

Computed by hand (regression R^2) since `statsmodels` isn't a project
dependency.

In [ ]:
AI23_COLS = [
    "employment_all_30min", "hospitals_30min", "gp_30min", "supermarkets_30min",
    "pharmacies_30min", "primary_schools_30min", "secondary_schools_30min", "main_bua_30min",
]
OSM_COLS = ["poi_residential", "poi_shopping", "poi_company", "poi_education",
            "poi_entertainment", "poi_scenic"]

ai23_log = np.log1p(feat[AI23_COLS])

def vif(df, col):
    y_ = df[col].values
    X_ = df.drop(columns=[col]).values
    X_ = np.column_stack([np.ones(len(X_)), X_])   # intercept
    beta, *_ = np.linalg.lstsq(X_, y_, rcond=None)
    resid = y_ - X_ @ beta
    ss_res = (resid ** 2).sum()
    ss_tot = ((y_ - y_.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    return 1 / (1 - r2)

vifs = {c: vif(ai23_log, c) for c in AI23_COLS}
vif_series = pd.Series(vifs).sort_values(ascending=False)
print("VIF (log1p AI23 block):")
print(vif_series.round(2))

vif_pharm, vif_gp = round(vifs["pharmacies_30min"], 2), round(vifs["gp_30min"], 2)
record("VIF pharmacies_30min", 37.98, vif_pharm, "" if vif_pharm == 37.98 else "MISMATCH")
record("VIF gp_30min", 18.24, vif_gp, "" if vif_gp == 18.24 else "MISMATCH")


In [ ]:
# strongest AI23-internal pair
corr = ai23_log.corr().abs()
corr_vals = corr.values.copy()
np.fill_diagonal(corr_vals, 0)
i, j = np.unravel_index(np.argmax(corr_vals), corr_vals.shape)
print(f"Strongest AI23-internal pair: {corr.index[i]} x {corr.columns[j]}, "
      f"|r| = {corr_vals[i, j]:.4f}")
strongest_r = round(float(corr_vals[i, j]), 2)
record("Strongest AI23-internal |r|", 0.98, strongest_r, "" if strongest_r == 0.98 else "MISMATCH")

# AI23 x OSM cross-correlation (48 pairs)
osm_log = np.log1p(feat[OSM_COLS])
cross = pd.DataFrame(index=AI23_COLS, columns=OSM_COLS, dtype=float)
for a in AI23_COLS:
    for o in OSM_COLS:
        cross.loc[a, o] = abs(np.corrcoef(ai23_log[a], osm_log[o])[0, 1])

max_pair = cross.stack().idxmax()
max_val = cross.stack().max()
print(f"\nMax |r| across the {cross.size} AI23xOSM pairs: {max_val:.4f}  (pair: {max_pair[0]} x {max_pair[1]})")
record("Max AI23xOSM |r| (48 pairs)", "\u2264 0.72 (claimed)", round(max_val, 4),
       "" if max_val <= 0.72 else "EXCEEDS STATED BOUND")


## 8. Spatial partition checks

### 8a. LSOA-to-borough nesting

Checks that LSOAs almost always sit inside a single borough (4,219 of
4,222) -- this is what lets leave-borough-out CV cleanly withhold every
LSOA in the test borough too.

In [ ]:
lsoa_to_boroughs = feat.groupby("lsoa11cd")["lad_name"].nunique()
n_lsoas = len(lsoa_to_boroughs)
n_nested = int((lsoa_to_boroughs == 1).sum())
pct_nested = 100 * n_nested / n_lsoas

print(f"Unique LSOAs: {n_lsoas:,}")
print(f"LSOAs mapping to exactly 1 borough: {n_nested:,} ({pct_nested:.2f}%)")

record_numeric("LSOAs total", 4222, n_lsoas)
record_numeric("LSOAs nested in a single borough (n)", 4219, n_nested)
record_numeric("LSOAs nested in a single borough (%)", 99.93, round(pct_nested, 2))


### 8b. Porosity graph

How many KNN edges cross a borough boundary. The dissertation says 89,715
edges total, 4,291 (4.8%) crossing, 2,019 stops (11.3%) affected.

The old `boundary_diagnostic_stops.csv` had different numbers and no
script behind it -- an orphaned file. This recomputes it from scratch and
matches the dissertation exactly, so it replaces that file as the source.

In [ ]:
from sklearn.neighbors import BallTree

lad_arr = feat["lad_name"].values
coords = np.radians(feat[["lat", "lon"]].values)

tree = BallTree(coords, metric="haversine")
_, nbrs = tree.query(coords, k=6)     # self + 5 nearest
nbr_idx = nbrs[:, 1:]                 # drop self -> shape (n, 5)

n = len(feat)
src = np.repeat(np.arange(n), 5)
dst = nbr_idx.reshape(-1)
cross_edge = lad_arr[src] != lad_arr[dst]

n_edges = len(src)
n_cross_edges = int(cross_edge.sum())
per_stop_cross = cross_edge.reshape(n, 5).sum(axis=1)
n_stops_any_cross = int((per_stop_cross >= 1).sum())
n_stops_all5_cross = int((per_stop_cross == 5).sum())

pct_cross_edges = 100 * n_cross_edges / n_edges
pct_stops_any = 100 * n_stops_any_cross / n
pct_stops_all5 = 100 * n_stops_all5_cross / n

print(f"total directed KNN edges: {n_edges:,} (expect {n*5:,})")
print(f"crossing edges: {n_cross_edges:,} ({pct_cross_edges:.2f}%)")
print(f"stops with >=1 cross neighbour: {n_stops_any_cross:,} ({pct_stops_any:.2f}%)")
print(f"stops with all 5 cross: {n_stops_all5_cross:,} ({pct_stops_all5:.2f}%)")

record_numeric("Porosity graph: total edges", 89715, n_edges)
record("Porosity graph: crossing edges (n)", 4291, n_cross_edges,
       "" if n_cross_edges == 4291 else "MISMATCH")
record("Porosity graph: crossing edges (%)", 4.8, round(pct_cross_edges, 1),
       "" if round(pct_cross_edges, 1) == 4.8 else "MISMATCH")
record("Porosity graph: stops with >=1 cross nbr (n)", 2019, n_stops_any_cross,
       "" if n_stops_any_cross == 2019 else "MISMATCH")
record("Porosity graph: stops with >=1 cross nbr (%)", 11.3, round(pct_stops_any, 1),
       "" if round(pct_stops_any, 1) == 11.3 else "MISMATCH")
record("Porosity graph: stops with all-5 cross (%)", 0.3, round(pct_stops_all5, 1),
       "" if round(pct_stops_all5, 1) == 0.3 else "MISMATCH")


## 9. Random forest importance of `main_bua_30min`

Backs up the claim that this feature barely matters: importance 0.0034
(std 0.0004), averaged over the 33 folds, AI23-only.

In [ ]:
rf_imp = pd.read_csv("results_rf_feature_importance.csv")
rf_imp = rf_imp.set_index(rf_imp.columns[0]) if "feature" not in rf_imp.columns else rf_imp.set_index("feature")
print(rf_imp)

row = rf_imp.loc["main_bua_30min"]
mean_col = [c for c in rf_imp.columns if "mean" in c.lower()][0]
std_col = [c for c in rf_imp.columns if "std" in c.lower()][0]
record_numeric("RF importance main_bua_30min (mean)", 0.0034, round(float(row[mean_col]), 4))
record_numeric("RF importance main_bua_30min (std)", 0.0004, round(float(row[std_col]), 4))


## Summary

Every check above, in one table. Anything flagged is a real mismatch
between the dissertation text and what the data actually shows -- as of
now, that's the two VIF values and the strongest AI23 pair (Section 7).
Fix the text, not the code -- this repo doesn't hold the manuscript, so
use the computed values above as the replacement.

In [ ]:
checks_df = pd.DataFrame(CHECKS)
checks_df["flag"] = checks_df["note"].apply(lambda n: "\u26a0" if n else "")
pd.set_option("display.max_rows", 200)
checks_df[["check", "dissertation", "computed", "flag"]]


In [ ]:
n_flagged = (checks_df["flag"] != "").sum()
print(f"{len(checks_df)} checks run, {n_flagged} flagged for review.")
if n_flagged:
    print()
    print(checks_df[checks_df["flag"] != ""][["check", "dissertation", "computed", "note"]].to_string(index=False))
